# Agente de Soporte al Cliente con PydanticAI + Ollama local

Version 2 de `pydanticai-101.ipynb`, adaptada para correr 100% en local con **Ollama**
(sin OpenAI API key, sin salida de datos del cliente a un tercero — soberania del dato
y soberania de la IA).

Cambios respecto al original:
- El `Agent` usa el endpoint OpenAI-compatible de Ollama (`http://localhost:11434/v1`),
  no la API de OpenAI.
- Modelo y parametros leidos de `config.ini` (no hardcodeados).
- CSV leido desde archivo local (`customer_support_tickets.csv`), no desde una URL externa.
- Se corrige `search_tickets` (en el notebook original la funcion se usaba pero nunca se
  definia, y el bloque de `prompt` tenia una indentacion invalida que rompia la celda).
- Se agrega una celda de prueba no interactiva (sin `input()`) para poder validar el
  agente durante desarrollo/CI, ademas del chat interactivo original.

**Requisito previo**: tener Ollama corriendo (`ollama serve`, normalmente ya activo como
servicio) y el modelo descargado (`ollama pull qwen2.5:7b-instruct-q4_K_M`).


In [12]:
import configparser
from pathlib import Path

config = configparser.ConfigParser()
config.read(Path.cwd() / "config.ini")

OLLAMA_BASE_URL = config.get("ollama", "base_url", fallback="http://localhost:11434/v1")
OLLAMA_MODEL = config.get("ollama", "modelo", fallback="qwen2.5:7b-instruct-q4_K_M")
OLLAMA_TEMPERATURE = config.getfloat("ollama", "temperature", fallback=0.1)
CSV_PATH = config.get("datos", "csv_path", fallback="customer_support_tickets.csv")
MAX_TICKETS_AL_LLM = config.getint("datos", "max_tickets_al_llm", fallback=10)

print(f"Ollama base_url : {OLLAMA_BASE_URL}")
print(f"Modelo          : {OLLAMA_MODEL}")
print(f"CSV             : {CSV_PATH}")

Ollama base_url : http://convergenciax02:11434
Modelo          : gemma4:26b
CSV             : customer_support_tickets.csv


### Instalacion (correr una vez, luego reiniciar el kernel)

In [2]:
#!pip install -r requirements.txt

### Librerias

In [14]:
import json
import pandas as pd
import nest_asyncio
from pydantic import BaseModel, Field, field_validator, ValidationError
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.ollama import OllamaProvider

nest_asyncio.apply()

### Dataset

Se lee el CSV local del proyecto (a diferencia del original, que lo bajaba de una URL
publica de IBM). Mantener los datos de clientes dentro de la carpeta del proyecto es
parte de la soberania del dato: nada sale a un bucket externo.

In [55]:
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head(5)

(8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


### Esquema de salida (`SupportResponse`)

Igual que en el original: fuerza que el LLM devuelva siempre una estructura valida
(categoria, prioridad, escalamiento, respuesta sugerida), sin importar que tan libre
sea el texto que genera el modelo local.

In [56]:
class SupportResponse(BaseModel):
    category: str = Field(..., description="The issue category")
    priority: str = Field(default="low", description="Issue priority (low, medium, high)")
    escalate: bool = Field(..., description="Should this be escalated to a human?")
    suggested_response: str = Field(..., description="A helpful support reply")

    @field_validator("priority", mode="before")
    @classmethod
    def validate_priority(cls, v):
        valid_priorities = {"low", "medium", "high"}
        print("SupportResponse->validate_priority"+ cls + "--- "+v)
        if isinstance(v, str) and v.lower() in valid_priorities:
            return v.lower()
        return "low"

    def priority_as_int(self) -> int:
        priority_map = {"low": 1, "medium": 2, "high": 3}
        print("SupportResponse->priority as int"+ self)
        return priority_map.get(self.priority, 1)

### Busqueda de tickets en el CSV

El original llamaba a `search_tickets(user_input, df)` sin definirla nunca. Aqui se
implementa: busca coincidencias por Ticket ID, email o nombre de cliente (case-insensitive,
substring), para no depender de que el usuario escriba el dato exacto.

In [57]:
def search_tickets(user_input: str, data: pd.DataFrame) -> pd.DataFrame:
    
    texto = user_input.strip().lower()
    print("search_tickets ->" + texto )
    if not texto:
        return data.iloc[0:0]

    columnas_email = "Customer Email"
    columnas_nombre = "Customer Name"
    columnas_id = "Ticket ID"

    mask_nombre = data[columnas_nombre].str.lower().str.contains(texto, na=False)
    mask_email = data[columnas_email].str.lower().str.contains(texto, na=False)
 

    # Ticket ID es numerico: solo intenta calzar si el texto (o un token de el) es un numero
    mask_id = pd.Series(False, index=data.index)
    for token in texto.replace(",", " ").split():
        if token.isdigit():
            mask_id = mask_id | (data[columnas_id] == int(token))
    print("search_tickets -> mask_nombre" + mask_nombre )
    print("search_tickets -> mask_email" + mask_email )
    print("search_tickets -> mask_id" + mask_id )
    return data[mask_nombre | mask_email | mask_id]

### Agente PydanticAI apuntando a Ollama local

`OllamaProvider` es el provider dedicado que trae `pydantic-ai` para Ollama (no requiere
api_key ni placeholders). El modelo se sigue construyendo con `OpenAIChatModel` porque
Ollama expone un servidor compatible con el protocolo de chat de OpenAI - es solo el
protocolo de red, nunca sale a internet: todo el trafico va a `localhost:11434`.

In [58]:
prompt = """
User: I forgot my password and can’t login.
Response:
{
  "category": "Authentication",
  "priority": "High",
  "escalate": true,
  "suggested_response": "You can reset your password from the login page. If that doesn’t work, we'll escalate this issue for manual support."
}

User: Please cancel my account.
Response:
{
  "category": "Subscription",
  "priority": "Medium",
  "escalate": false,
  "suggested_response": "To cancel your subscription, go to Settings > Billing > Cancel Plan. Contact us if you need help."
}
"""

In [50]:
ollama_model = OpenAIChatModel(
    model_name=OLLAMA_MODEL,
    provider=OllamaProvider(base_url=OLLAMA_BASE_URL),
)

agent = Agent(
    ollama_model,
    output_type=SupportResponse,
    model_settings={"temperature": OLLAMA_TEMPERATURE},
    instructions=f"""
        You are a helpful customer support assistant.
        You will receive a customer request and, when available, matching rows from 
        the customer support ticket database in JSON.
        Rules:
        - Respond ONLY using the data provided in the matching ticket JSON, if any.
        - If no matching ticket data is given, respond politely asking the user to "
        check their ticket ID, email, or name.
        - Output MUST follow the SupportResponse structure exactly.
        
        You can use this CSV data:
        {df.to_string(index=False)}

        Your task is to:
        - Match user input to any ticket using ticket_id, email, or customer_name."
        - Respond ONLY using the data in the CSV."
        - If no match is found, respond politely asking the user to check their input.""
        - Output MUST follow the SupportResponse structure."""
        
    ,
)

### Prueba no interactiva

Util para validar que el agente + Ollama responden correctamente sin necesitar `input()`
(por ejemplo al ejecutar el notebook en batch).

In [51]:
async def probar_agente(mensaje: str):
    matches = search_tickets(mensaje, df).head(MAX_TICKETS_AL_LLM)
    print("probar_agente->matches:"+ matches)
    if matches.empty:
        prompt = f"Customer request:\n\n{mensaje}\n\nNo matching ticket data was found."
    else:
        ticket_data = matches.to_json(orient="records", indent=2)
        prompt = (
            f"Customer request:\n\n{mensaje}\n\n"
            f"Matching ticket data:\n\n{ticket_data}\n\n"
            "Using ONLY the data above, generate the SupportResponse."
        )
    result = await agent.run(prompt)
    return result.output



In [52]:
print(probar_agente("Christopher Robbins purchased a Dell XPS. If you need more details, please let us know"))
print(probar_agente("Christopher Robbins purchased a Dell XPS?"))

<coroutine object probar_agente at 0x000001BAECC07740>
<coroutine object probar_agente at 0x000001BAECC04940>


C:\Users\Joel\AppData\Local\Temp\ipykernel_12692\3708084061.py:1: RuntimeWarning: coroutine 'probar_agente' was never awaited
  print(probar_agente("Christopher Robbins purchased a Dell XPS. If you need more details, please let us know"))
C:\Users\Joel\AppData\Local\Temp\ipykernel_12692\3708084061.py:2: RuntimeWarning: coroutine 'probar_agente' was never awaited
  print(probar_agente("Christopher Robbins purchased a Dell XPS?"))


### Chat interactivo (igual espiritu que el notebook original, corregido)

In [53]:
import json
from pydantic import ValidationError

async def run_chatbot():
    print(" Customer Support Chatbot — type 'exit' to quit\n")

    while True:
        user_input = input("🧑 You: ").strip()
        if user_input.lower() == "exit":
            print(" Exiting chatbot.")
            break

        try:
            result = await agent.run(user_input)
            output = result.output

            # Try parsing the result into SupportResponse
            if isinstance(output, SupportResponse):
                response = output
            elif isinstance(output, dict):
                response = SupportResponse(**output)
            elif isinstance(output, str):
                response = SupportResponse(**json.loads(output))
            else:
                raise ValueError("Unsupported output format from agent.")

            # ✅ Print structured response
            print("\nAI Response:")
            print(f"Category        : {response.category}")
            print(f"Priority        : {response.priority}")
            print(f"Escalate        : {'Yes' if response.escalate else 'No'}")
            print(f"Suggested Reply : {response.suggested_response}\n")

        except (ValidationError, ValueError, json.JSONDecodeError) as e:
            print("\n❌ Could not parse structured output from agent:")
            print(output)

# Run this to launch chatbot
await run_chatbot()

 Customer Support Chatbot — type 'exit' to quit



NoEventLoopError: Not currently running on any asynchronous event loop. Available async backends: asyncio, trio

In [54]:
async def run_chatbot():
    print("Customer Support Chatbot (Ollama local) — escribe 'exit' para salir\n")

    while True:
        user_input = input(" You: ").strip()
        if user_input.lower() == "exit":
            print("Exiting chatbot.")
            break
        if not user_input:
            continue

        try:
            
            # Try parsing the result into SupportResponse
            if isinstance(user_input, SupportResponse):
                response = user_input
            elif isinstance(user_input, dict):
                response = SupportResponse(**user_input)
            elif isinstance(user_input, str):
                response = SupportResponse(**json.loads(user_input))
            else:
                raise ValueError("Unsupported output format from agent.")
            
            # ✅ Print structured response
            print("\nAI Response:")
            print(f"Category        : {response.category}")
            print(f"Priority        : {response.priority}")
            print(f"Escalate        : {'Yes' if response.escalate else 'No'}")
            print(f"Suggested Reply : {response.suggested_response}\n")
            
            matches = search_tickets(user_input, df)

            if matches.empty:
                print("\nAI Response:")
                print("No matching ticket was found. Please check the ticket ID, email, or customer name.\n")
                continue

            matches = matches.head(MAX_TICKETS_AL_LLM)
            ticket_data = matches.to_json(orient="records", indent=2)
            prompt = (
                f"Customer request:\n\n{user_input}\n\n"
                f"Matching ticket data:\n\n{ticket_data}\n\n"
                "Using ONLY the data above, generate the SupportResponse."
            )

            result = await agent.run(prompt)
            response = result.output

            print("\nAI Response:")
            print(f"Category        : {response.category}")
            print(f"Priority        : {response.priority}")
            print(f"Escalate        : {'Yes' if response.escalate else 'No'}")
            print(f"Suggested Reply : {response.suggested_response}\n")

        except (ValidationError, ValueError, json.JSONDecodeError) as e:
            print(f"\n Could not parse structured output from agent: {type(e).__name__}: {e}\n")
        except Exception as e:
            print(f"\n Error: {type(e).__name__}: {e}\n")

# Ejecutar para lanzar el chat
await run_chatbot()

Customer Support Chatbot (Ollama local) — escribe 'exit' para salir


 Could not parse structured output from agent: JSONDecodeError: Expecting value: line 1 column 1 (char 0)


 Could not parse structured output from agent: JSONDecodeError: Expecting value: line 1 column 1 (char 0)

Exiting chatbot.


## Preguntas de prueba sugeridas

- `Christopher Robbins purchased a Dell XPS. If you need more details, please let us know.`
- `My Dell XPS keeps dropping from Wi-Fi every few minutes.`
- `Ticket 4`
